# Twitter-RoBERTa Experiments for Sentiment Classification

## Purpose

This notebook explores the performance of a **domain-specific transformer model (Twitter-RoBERTa)** for binary sentiment classification.

Unlike general-purpose models (e.g., DistilBERT), Twitter-RoBERTa is pre-trained on **Twitter-specific text**, making it more suitable for handling:
- Slang and informal language  
- Hashtags and mentions  
- Emojis and noisy social media text  

The goal is to evaluate whether **domain-specific pretraining improves performance and data efficiency** in sentiment analysis tasks.

---

## What This Notebook Does

- Trains Twitter-RoBERTa on sentiment datasets of varying sizes:
  - Small (~100k samples)
  - Medium (~200k samples)
  - Large (~400k samples)

- Keeps experimental settings consistent with previous DistilBERT experiments:
  - Same train/test split
  - Same evaluation metrics
  - Comparable hyperparameters

- Evaluates performance using:
  - Accuracy  
  - F1-score  
  - ROC-AUC  
  - Training time  

---

## Key Research Focus

This experiment is designed to answer:

- Does a **domain-specific transformer outperform a general-purpose model**?
- Does Twitter-RoBERTa achieve better results with **less training data**?
- How does it compare to DistilBERT in terms of:
  - Performance scaling  
  - Efficiency  
  - Stability  

---

## Role in Overall Project

This notebook is a key extension of the baseline study and will be used to:

- Compare against **DistilBERT dataset scaling results**
- Analyze **data efficiency across model architectures**
- Evaluate the impact of **domain-specific pretraining**
- Support further extension into **brand sentiment analysis and domain adaptation**

---

## Expected Contribution

This experiment aims to provide insights into:

> Whether domain-specific transformer models reduce the need for large datasets while improving sentiment classification performance on social-media-style text.

In [1]:
!uv pip install transformers datasets tqdm accelerate

Using Python 3.12.12 environment at: /usr
Audited 4 packages in 545ms


In [2]:
# !pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -U transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 111.4 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling trans

Checking GPU

In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
# !nvidia-smi

In [5]:
# ! pip install transformers[torch] datasets tqdm accelerate --only-binary :all: -i https://pypi.tuna.tsinghua.edu.cn/simple

# Dependencies

In [6]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline, BertTokenizerFast
from transformers import (
    DistilBertForSequenceClassification, 
    DistilBertTokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
import time
import os
import json, pickle as pkl



# Loading Dataset

In [7]:
# ! pip install kaggle

get kaggle.json from the kaggle and add to working environment

In [8]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

Importing twitter sentiment dataset

In [9]:
# !kaggle datasets download -d kazanova/sentiment140

if data set is downloaded using: !kaggle datasets download -d kazanova/sentiment140

In [10]:
# # extracting the compressed dataset

# from zipfile import ZipFile
# dataset = '/content/sentiment140.zip'

# with ZipFile(dataset, 'r') as zip:
#   zip.extractall()
#   print('The dataset is extracted successfully')

Use datasets/kazanova/sentiment140 from kaggle

In [11]:
# colab
# df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

#colab + dataset saved on drive
# df = pd.read_csv('/content/drive/MyDrive/Sentiment Analysis Project/datasets/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

#kaggle
df = pd.read_csv(
    '/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv',
    encoding='ISO-8859-1',
    header=None
)

df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

# Converting labels
df['target'] = df['target'].replace(4,1)

# Keeping only needed columns
df = df[['text', 'target']]


## Create 3 datasets

- Dataset 1: Small (50k per class → 100k total)
- Dataset 2: Medium (100k per class → 200k total) MAIN
- Dataset 3: Large (200k per class → 400k total)

In [12]:
df_small = df.groupby('target').sample(50000, random_state=42).reset_index(drop=True)

In [13]:
df_medium = df.groupby('target').sample(100000, random_state=42).reset_index(drop=True)

In [14]:
df_large = df.groupby('target').sample(200000, random_state=42).reset_index(drop=True)

Sanity check

In [15]:
print("Small:\n", df_small['target'].value_counts())
print("\nMedium:\n", df_medium['target'].value_counts())
print("\nLarge:\n", df_large['target'].value_counts())

Small:
 target
0    50000
1    50000
Name: count, dtype: int64

Medium:
 target
0    100000
1    100000
Name: count, dtype: int64

Large:
 target
0    200000
1    200000
Name: count, dtype: int64


In [16]:
# Uncomment below code to save the datasets

# df_small.to_csv("sentiment_small.csv", index=False)
# df_medium.to_csv("sentiment_medium.csv", index=False)
# df_large.to_csv("sentiment_large.csv", index=False)

# df_small.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_small.csv", index=False)
# df_medium.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_medium.csv", index=False)
# df_large.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_large.csv", index=False)

# REUSABLE Twitter-RoBERTa EXPERIMENT PIPELINE

## Metrics function

In [17]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)

  precision, recall,f1, _ = precision_recall_fscore_support(labels, preds, average = 'binary')
  acc = accuracy_score(labels, preds)

  probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
  roc = roc_auc_score(labels, probs)

  return {
      "accuracy": acc,
      "f1": f1,
      "roc_auc": roc,
      "precision": precision,
      "recall": recall
  }




MAIN reusable function

kaggle

In [18]:
def run_experiment(
    df,
    dataset_name="dataset",
    save_dir="/kaggle/working/models",
    model_name="cardiffnlp/twitter-roberta-base-sentiment"
):

    print(f"Running experiment on {dataset_name} using {model_name}")

    # Save directory logic
    exp_path = os.path.join(save_dir, dataset_name)
    os.makedirs(exp_path, exist_ok=True)

    
    # Train / Validation Split
    
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text'].tolist(),
        df['target'].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df['target']
    )

    with open(os.path.join(exp_path, "data_split_info.json"), "w") as f:
        json.dump({
            "train_size": len(train_texts),
            "val_size": len(val_texts)
        }, f, indent=4)

    
    # Tokenizer 
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    val_encodings = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    
    # Dataset Class
    
    class SentimentDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item

        def __len__(self):
            return len(self.labels)

    train_dataset = SentimentDataset(train_encodings, train_labels)
    val_dataset = SentimentDataset(val_encodings, val_labels)

    
    # Model 
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    
    # Training Arguments
    
    training_args = TrainingArguments(
        output_dir=os.path.join(exp_path, "checkpoints"),
        num_train_epochs=2,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
    
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
    
        eval_strategy="no",
        save_strategy="no",
        load_best_model_at_end=False,
    
        fp16=True,
        optim="adamw_torch",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        logging_steps=200,
        report_to="none"
    )

    
    # Trainer
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    
    # Training
    
    start = time.time()
    trainer.train()
    end = time.time()

    trainer.save_state()

    
    # Evaluation
    
    results = trainer.evaluate()

    final_results = {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": results.get("eval_accuracy"),
        "f1": results.get("eval_f1"),
        "roc_auc": results.get("eval_roc_auc"),
        "training_time_sec": end - start
    }

    
    # Save Outputs
    
    trainer.save_model(exp_path)
    tokenizer.save_pretrained(exp_path)

    with open(os.path.join(exp_path, "results.json"), "w") as f:
        json.dump(final_results, f, indent=4)

    with open(os.path.join(exp_path, "results.pkl"), "wb") as f:
        pkl.dump(final_results, f)

    with open(os.path.join(exp_path, "training_args.json"), "w") as f:
        json.dump(training_args.to_dict(), f, indent=4)

    label_map = {0: "negative", 1: "positive"}
    with open(os.path.join(exp_path, "label_map.json"), "w") as f:
        json.dump(label_map, f, indent=4)

    print(f"\nExperiment saved at: {exp_path}")

    print("\n--- Files Generated ---")
    for root, dirs, files in os.walk(exp_path):
        for file in files:
            print(os.path.join(root, file))

    return final_results

colab

# Run all experiments

In [19]:
import transformers
print(transformers.__version__)

5.7.0


In [20]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

CUDA available: True
GPU name: Tesla T4


In [21]:
# !nvidia-smi

In [22]:
results_medium = run_experiment(
    df_medium,
    dataset_name="df_medium",
    model_name="cardiffnlp/twitter-roberta-base-sentiment"
)

Running experiment on df_medium using cardiffnlp/twitter-roberta-base-sentiment


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,2.895564
400,2.449302
600,2.370341
800,2.122849
1000,2.045974
1200,2.069929


Training Loss,Validation Loss,Step,Accuracy,F1,Roc Auc,Precision,Recall
2.069929,0.558872,1250,0.886825,0.886236,0.954941,0.890871,0.881650


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_medium

--- Files Generated ---
/kaggle/working/models/df_medium/results.json
/kaggle/working/models/df_medium/label_map.json
/kaggle/working/models/df_medium/training_args.json
/kaggle/working/models/df_medium/tokenizer_config.json
/kaggle/working/models/df_medium/model.safetensors
/kaggle/working/models/df_medium/tokenizer.json
/kaggle/working/models/df_medium/training_args.bin
/kaggle/working/models/df_medium/data_split_info.json
/kaggle/working/models/df_medium/config.json
/kaggle/working/models/df_medium/results.pkl
/kaggle/working/models/df_medium/checkpoints/trainer_state.json


In [23]:
# results_large = run_experiment(df_large, "large_200k")
results_large = run_experiment(
    df_large,
    dataset_name="df_large",
    model_name="cardiffnlp/twitter-roberta-base-sentiment"
)
results_large

Running experiment on df_large using cardiffnlp/twitter-roberta-base-sentiment


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,2.937020
400,2.484122
600,2.354599
800,2.276350
1000,2.222883
1200,2.264394
1400,2.059546
1600,2.001146
1800,1.998960
2000,1.964639


Training Loss,Validation Loss,Step,Accuracy,F1,Roc Auc,Precision,Recall
1.984314,0.542506,2500,0.890537,0.889878,0.957802,0.895271,0.884550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_large

--- Files Generated ---
/kaggle/working/models/df_large/results.json
/kaggle/working/models/df_large/label_map.json
/kaggle/working/models/df_large/training_args.json
/kaggle/working/models/df_large/tokenizer_config.json
/kaggle/working/models/df_large/model.safetensors
/kaggle/working/models/df_large/tokenizer.json
/kaggle/working/models/df_large/training_args.bin
/kaggle/working/models/df_large/data_split_info.json
/kaggle/working/models/df_large/config.json
/kaggle/working/models/df_large/results.pkl
/kaggle/working/models/df_large/checkpoints/trainer_state.json


{'dataset': 'df_large',
 'model': 'cardiffnlp/twitter-roberta-base-sentiment',
 'accuracy': 0.8905375,
 'f1': 0.8898781453955559,
 'roc_auc': 0.95780231,
 'training_time_sec': 7636.226632118225}

In [24]:
results_small = run_experiment(
    df_small,
    dataset_name="df_small",
    model_name="cardiffnlp/twitter-roberta-base-sentiment"
)
results_small

Running experiment on df_small using cardiffnlp/twitter-roberta-base-sentiment


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,2.901288
400,2.344419
600,2.121015


Training Loss,Validation Loss,Step,Accuracy,F1,Roc Auc,Precision,Recall
2.121015,0.599367,626,0.877050,0.877044,0.948672,0.877088,0.877000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_small

--- Files Generated ---
/kaggle/working/models/df_small/results.json
/kaggle/working/models/df_small/label_map.json
/kaggle/working/models/df_small/training_args.json
/kaggle/working/models/df_small/tokenizer_config.json
/kaggle/working/models/df_small/model.safetensors
/kaggle/working/models/df_small/tokenizer.json
/kaggle/working/models/df_small/training_args.bin
/kaggle/working/models/df_small/data_split_info.json
/kaggle/working/models/df_small/config.json
/kaggle/working/models/df_small/results.pkl
/kaggle/working/models/df_small/checkpoints/trainer_state.json


{'dataset': 'df_small',
 'model': 'cardiffnlp/twitter-roberta-base-sentiment',
 'accuracy': 0.87705,
 'f1': 0.8770438521926096,
 'roc_auc': 0.9486719699999999,
 'training_time_sec': 1909.6633615493774}

# Compare results

In [25]:
results_df = pd.DataFrame([
    results_small,
    results_medium,
    results_large
])

results_df

,dataset,model,accuracy,f1,roc_auc,training_time_sec
0,df_small,cardiffnlp/twitter-roberta-base-sentiment,0.877050,0.877044,0.948672,1909.663362
1,df_medium,cardiffnlp/twitter-roberta-base-sentiment,0.886825,0.886236,0.954941,3808.744193
2,df_large,cardiffnlp/twitter-roberta-base-sentiment,0.890537,0.889878,0.957802,7636.226632


# Plots

Load Results Automatically

In [26]:
# import json
# import matplotlib.pyplot as plt

# EXPERIMENT_PATHS = {
#     "small": "/kaggle/input/datasets/aayushparajuli03/small-result",
#     "medium": "/kaggle/input/notebooks/aayushparajuli03/sentiment-analysis/models/df_medium/results.json",
#     "large": "/kaggle/input/notebooks/aayushparajuli03/sentiment-analysis/models/large_200k/results.json",
# }

In [27]:
# import os
# import json

# def load_result(path):
#     """
#     Loads a single results.json from either:
#     - direct file path
#     - folder containing results.json
#     """
#     if os.path.isdir(path):
#         path = os.path.join(path, "results.json")

#     if not os.path.exists(path):
#         print(f"Missing: {path}")
#         return None

#     with open(path, "r") as f:
#         data = json.load(f)

#     return data


# def load_all_results(paths_dict):
#     results = []

#     for name, path in paths_dict.items():
#         data = load_result(path)
#         if data:
#             data["dataset"] = name   # enforce consistent label
#             results.append(data)

#     return results

Convert to Structured Format

In [28]:
# def prepare_data(results):
#     results = sorted(results, key=lambda x: x["training_time_sec"])

#     datasets = [r["dataset"] for r in results]
#     accuracy = [r["accuracy"] for r in results]
#     f1 = [r["f1"] for r in results]
#     roc_auc = [r["roc_auc"] for r in results]
#     time = [r["training_time_sec"] for r in results]

#     return datasets, accuracy, f1, roc_auc, time

Plot Functions

In [29]:
# def plot_metric(x, y, title, xlabel, ylabel):
#     plt.figure()
#     plt.plot(x, y, marker='o')
#     plt.xlabel(xlabel)
#     plt.ylabel(ylabel)
#     plt.title(title)
#     plt.grid()
#     plt.show()


# def plot_all(datasets, accuracy, f1, roc_auc, time):
#     plot_metric(datasets, accuracy, "Dataset vs Accuracy", "Dataset", "Accuracy")
#     plot_metric(datasets, f1, "Dataset vs F1 Score", "Dataset", "F1 Score")
#     plot_metric(datasets, roc_auc, "Dataset vs ROC-AUC", "Dataset", "ROC-AUC")
#     plot_metric(datasets, time, "Dataset vs Training Time", "Dataset", "Time (sec)")

Dataset vs ROC-AUC

In [30]:
# def plot_roc_auc(datasets, roc_auc):
#     plt.figure()
#     plt.plot(datasets, roc_auc, marker='o')
#     plt.xlabel("Dataset")
#     plt.ylabel("ROC-AUC")
#     plt.title("Dataset vs ROC-AUC")
#     plt.grid()
#     plt.show()

Training Time vs Accuracy

In [31]:
# def plot_time_vs_accuracy(time, accuracy):
#     plt.figure()
#     plt.plot(time, accuracy, marker='o')

#     for i, val in enumerate(accuracy):
#         plt.text(time[i], val, f"{val:.3f}", ha='center')

#     plt.xlabel("Training Time (sec)")
#     plt.ylabel("Accuracy")
#     plt.title("Training Time vs Accuracy")
#     plt.grid()
#     plt.show()

Add annotations

In [32]:
# def plot_accuracy_with_values(datasets, accuracy):
#     plt.figure()
#     plt.plot(datasets, accuracy, marker='o')

#     for i, val in enumerate(accuracy):
#         plt.text(i, val, f"{val:.3f}", ha='center')

#     plt.xlabel("Dataset")
#     plt.ylabel("Accuracy")
#     plt.title("Dataset vs Accuracy")
#     plt.grid()
#     plt.show()

Run 

In [33]:
# plt.close('all')
# plot_accuracy_with_values(datasets, accuracy)

# plot_metric(datasets, f1, "Dataset vs F1 Score", "Dataset", "F1 Score")

# plot_metric(datasets, roc_auc, "Dataset vs ROC-AUC", "Dataset", "ROC-AUC")

# plot_time_vs_accuracy(time, accuracy)

# Impact of Dataset Scaling on Transformer-Based Sentiment Classification  
### A Comparative Study using DistilBERT

---

## Abstract
This study investigates the effect of dataset scaling on the performance of transformer-based models for binary sentiment classification. Using DistilBERT as the baseline architecture, we conduct controlled experiments across three dataset sizes to evaluate performance improvements in terms of Accuracy, F1-score, ROC-AUC, and training efficiency. Results indicate that increasing dataset size yields consistent but diminishing performance gains, while significantly increasing computational cost. The findings highlight the trade-off between data volume and model efficiency in practical machine learning systems.

---

## 1. Introduction
Transformer-based architectures have become the dominant paradigm in Natural Language Processing (NLP), with models such as BERT and its variants achieving state-of-the-art results across multiple tasks. However, the relationship between dataset size and model performance—particularly under constrained computational budgets—remains an important practical consideration.

This research aims to empirically analyze:
- How performance scales with increasing dataset size
- The trade-offs between training time and predictive performance
- The stability and convergence behavior of the model

---

## 2. Methodology

### 2.1 Model Architecture
We utilize **DistilBERT (distilbert-base-uncased)**, a compressed version of BERT that retains ~97% of its performance while reducing computational overhead.

- Task: Binary Sentiment Classification  
- Output Classes: {0: Negative, 1: Positive}  

---

### 2.2 Experimental Design

To isolate the impact of dataset size, all experiments were conducted under identical training configurations, varying only the dataset size.

#### Datasets:
| Dataset | Description |
|--------|------------|
| df_small | Small-scale dataset(~100k samples) |
| df_medium | Medium-scale dataset(~200k samples) |
| df_large | Large-scale dataset (~400k samples) |

---

### 2.3 Training Configuration

| Parameter | Value |
|----------|------|
| Epochs | 2 |
| Batch Size (per device) | 32 |
| Gradient Accumulation | 4 |
| Effective Batch Size | 256 |
| Learning Rate | 2e-5 |
| Optimizer | AdamW |
| Precision | FP16 |
| Max Sequence Length | 128 |
| Train/Validation Split | 80/20 (Stratified) |

---

### 2.4 Evaluation Metrics
The following metrics were used:
- **Accuracy**: Overall correctness
- **F1 Score**: Balance between precision and recall
- **ROC-AUC**: Ranking capability across thresholds
- **Training Time**: Computational cost indicator

---

## 3. Results

### 3.1 Performance Comparison

| Dataset | Accuracy | F1 Score | ROC-AUC | Training Time (sec) |
|--------|---------|---------|--------|--------------------|
| df_small | 0.8377 | 0.8315 | 0.9203 | 1319 |
| df_medium | 0.8426 | 0.8427 | 0.9220 | 1852 |
| large_200k | 0.8515 | 0.8511 | 0.9294 | 3709 |

---

### 3.2 Observations

#### 3.2.1 Performance Scaling
- Accuracy improves steadily from **0.8377 → 0.8515**
- F1 Score shows similar gains, indicating balanced class predictions
- ROC-AUC improves consistently, suggesting better decision boundary separation

#### 3.2.2 Diminishing Returns
- Performance gains reduce as dataset size increases:
  - Small → Medium: noticeable improvement
  - Medium → Large: smaller incremental gain
- Indicates **non-linear scaling behavior**

---

### 3.3 Training Dynamics

#### Medium Dataset:
- Smooth loss reduction from **3.62 → 2.71**
- Stable convergence behavior

#### Large Dataset:
- More fluctuations during training
- Final loss lower (**2.60**), indicating better fit
- Suggests increased robustness but higher variance during optimization

---

## 4. Discussion

### 4.1 Impact of Dataset Size
The results confirm that larger datasets improve generalization performance. This aligns with established deep learning theory, where increased data exposure enables better representation learning.

However:
- Gains are **incremental, not exponential**
- Larger datasets primarily refine decision boundaries rather than drastically improving performance

---

### 4.2 Computational Trade-offs

| Transition | Accuracy Gain | Time Increase |
|-----------|-------------|--------------|
| Small → Medium | +0.5% | +40% |
| Medium → Large | +0.9% | +100% |

Key Insight:
> Doubling training time does not proportionally improve model performance.

This highlights a critical constraint in real-world systems:
- Resource efficiency must be balanced with marginal performance gains

---

### 4.3 Model Stability and Convergence
- Larger datasets result in:
  - Lower validation loss
  - Improved ROC-AUC
- However, they introduce:
  - Higher training variance
  - Longer convergence time

This suggests that:
- Larger datasets improve **generalization**
- But require better optimization strategies (e.g., scheduling, early stopping)

---

### 4.4 Practical Implications

For practitioners:
- Medium-sized datasets may offer the best **cost-performance trade-off**
- Large datasets are beneficial when:
  - High precision is critical
  - Computational resources are available

---

## 5. Limitations

- Only 2 training epochs were used (underfitting possible)
- No hyperparameter tuning performed
- Evaluation during training was disabled
- Single model architecture used (no baseline comparison)

---

## 6. Future Work

- Extend training epochs (3–5) to study convergence
- Perform hyperparameter optimization
- Compare with larger models (BERT, RoBERTa)
- Introduce learning rate schedulers
- Analyze impact of sequence length (128 vs 256)
- Incorporate early stopping mechanisms

---

## 7. Conclusion

This study demonstrates that increasing dataset size improves the performance of transformer-based models for sentiment classification. However, the improvement follows a pattern of diminishing returns, where computational cost grows faster than performance gains.

The findings emphasize the importance of:
- Strategic dataset scaling
- Efficient resource utilization
- Balanced model optimization

In practical deployments, selecting an optimal dataset size is crucial to achieving high performance without incurring unnecessary computational expense.

---

## 8. Reproducibility

- Framework: Hugging Face Transformers  
- Model: DistilBERT  
- Hardware: GPU (T4, mixed precision enabled)  
- Code includes:
  - Data preprocessing
  - Tokenization
  - Training pipeline
  - Evaluation and logging

In [34]:
# !git config --global user.name "aayush-12321"
# !git config --global user.email "aayushparajuli23@gmail.com"

In [35]:
!